# Bigram + Location Features Results Viewer
ดูผลลัพธ์จาก `bigram_full_pipeline.py` เวอร์ชันที่เพิ่ม location features (`same_country`, `same_city`, `geo_distance_km`)


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)
base = Path('data/processed/bigram_pipeline_results_with_location')
if not base.exists():
    raise FileNotFoundError(f'ไม่พบโฟลเดอร์ผลลัพธ์: {base}')
print('using', base)


In [ ]:
metrics_path = base / 'bigram_pipeline_metrics_summary.csv'
metrics = pd.read_csv(metrics_path)
metrics


In [ ]:
acc = metrics.pivot(index='pair_name', columns='model', values='top1_accuracy').sort_index()
rec = metrics.pivot(index='pair_name', columns='model', values='candidate_recall_topk').sort_index()
print('Top-1 Accuracy')
display(acc)
print('Candidate Recall@top-k')
display(rec)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for model, g in metrics.groupby('model'):
    g = g.sort_values('pair_name')
    axes[0].plot(g['pair_name'], g['top1_accuracy'], marker='o', label=model)
    axes[1].plot(g['pair_name'], g['candidate_recall_topk'], marker='o', label=model)
axes[0].set_title('Top-1 Accuracy')
axes[1].set_title('Candidate Recall@top-k')
axes[0].tick_params(axis='x', rotation=15)
axes[1].tick_params(axis='x', rotation=15)
axes[0].legend(); axes[1].legend()
plt.tight_layout(); plt.show()


In [ ]:
# โหลด prediction files ทั้งหมด (SGD/Linear + RF)
pred_files = sorted(base.glob('*_top1_predictions_*.csv'))
print('prediction files:', len(pred_files))
for p in pred_files:
    print('-', p.name)
preds = {p.name: pd.read_csv(p) for p in pred_files}


In [ ]:
# เลือกไฟล์ที่ต้องการดู
file_name = sorted(preds.keys())[0]
df = preds[file_name]
print('selected:', file_name, 'rows=', len(df))
df.head(10)


In [ ]:
# ดูผลลัพธ์เฉพาะคอลัมน์สำคัญ + location features
cols = [c for c in [
    'source_profile_id','predicted_profile_id','is_correct_top1',
    'predicted_match_probability','uu_sim','un_sim','ub_sim',
    'same_country','same_city','geo_distance_km','true_candidate_in_topk'
] if c in df.columns]
display(df[cols].head(20))


In [ ]:
# เปรียบเทียบ feature เฉลี่ยระหว่างทายถูก vs ทายผิด
agg_cols = [c for c in ['uu_sim','un_sim','ub_sim','same_country','same_city','geo_distance_km'] if c in df.columns]
cmp = df.groupby('is_correct_top1')[agg_cols].mean(numeric_only=True)
cmp


In [ ]:
# Distribution ของ geo distance
if 'geo_distance_km' in df.columns:
    plt.figure(figsize=(8,4))
    df['geo_distance_km'].clip(upper=5000).hist(bins=40)
    plt.title('Geo Distance Distribution (clipped at 5000 km)')
    plt.xlabel('geo_distance_km')
    plt.ylabel('count')
    plt.show()
else:
    print('no geo_distance_km column')
